# Lesson 02 - Microsoft Agent Framework का अन्वेषण

**Microsoft Agent Framework (MAF)** एक एकीकृत ढांचा है जो AI एजेंट बनाने के लिए है। यह एक साफ, संरचित वास्तुकला प्रदान करता है जिसमें चार मूल निर्माण खंड होते हैं:

- **Client** – एक AI मॉडल एंडपॉइंट से जुड़ता है और संचार को संभालता है
- **Agent** – क्लाइंट को निर्देशों और टूल परिभाषाओं के साथ लपेटता है
- **Tools** – एजेंट की क्षमताओं को कस्टम फ़ंक्शंस के साथ बढ़ाता है जिन्हें मॉडल कॉल कर सकता है
- **Session** – बहु-चरणीय संवादों के लिए बातचीत के इतिहास को बनाए रखता है

इस पाठ में, हम इन अवधारणाओं का उपयोग करके एक **यात्रा बुकिंग एजेंट** बनाएंगे जो गंतव्य की उपलब्धता की जाँच करता है।


## सेटअप


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## एजेंट फ्रेमवर्क आर्किटेक्चर को समझना

Microsoft एजेंट फ्रेमवर्क एक परतदार आर्किटेक्चर का पालन करता है:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **क्लाइंट** – एक `AzureAIProjectAgentProvider` Azure OpenAI तैनाती से जुड़ता है। यह प्रमाणीकरण, अनुरोध प्रारूपण, और प्रतिक्रिया पार्सिंग को संभालता है।
2. **एजेंट** – क्लाइंट से `provider.create_agent()` के माध्यम से बनाया गया, एजेंट मॉडल एक्सेस को निर्देशों (सिस्टम प्रॉम्प्ट) और टूल्स के साथ जोड़ता है।
3. **टूल्स** – पायथन फ़ंक्शन जिन्हें `@tool` से सजाया गया है जिनका उपयोग एजेंट क्रियाएँ करने या डेटा प्राप्त करने के लिए कर सकता है।
4. **सेशन** – एक `AgentSession` ऑब्जेक्ट (जो `agent.create_session()` के माध्यम से बनाया गया है) जो वार्तालाप इतिहास संग्रहीत करता है, जिससे मल्टी-टर्न संवाद संभव होता है जहां एजेंट पिछला संदर्भ याद रखता है।

आइए प्रत्येक परत को चरण दर चरण बनाएं।


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool डेकोरेटर के साथ टूल जोड़ना

टूल एजेंट्स को टेक्स्ट जनरेट करने से परे क्रियाएं करने देता है। `@tool` डेकोरेटर एक सामान्य पाइथन फंक्शन को कुछ ऐसा में बदल देता है जिसे एजेंट कॉल कर सकता है।

मुख्य बिंदु:
- मॉडल को प्रत्येक पैरामीटर समझाने के लिए `Annotated[type, "description"]` का उपयोग करें।
- डॉकस्ट्रिंग वह टूल विवरण बन जाती है जिसे मॉडल देखता है।
- `approval_mode="never_require"` का मतलब है कि टूल स्वचालित रूप से बिना उपयोगकर्ता की पुष्टि के चलता है।


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## उपकरणों के साथ एक एजेंट बनाना

अब हम क्लाइंट, निर्देशों, और उपकरणों को एक एजेंट में संयोजित करते हैं। `instructions` सिस्टम प्रॉम्प्ट के रूप में कार्य करते हैं — वे एजेंट की व्यक्तित्व और व्यवहार को परिभाषित करते हैं।


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## सत्रों के साथ बहु-वार्ता बातचीत

एक `AgentSession` (जो `agent.create_session()` के माध्यम से बनाया जाता है) बातचीत में सभी संदेशों को ट्रैक रखता है। एक ही सत्र को प्रत्येक `agent.run()` कॉल में पास करके, एजेंट के पास पूरी बातचीत का इतिहास होता है और वह पहले के संदेशों का संदर्भ दे सकता है।

हम `tools=[check_destination_availability]` पास करते हैं ताकि एजेंट हर चरण में हमारे उपलब्धता जांचकर्ता को कॉल कर सके।


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## सारांश

इस पाठ में आपने Microsoft Agent Framework के चार स्तंभों का पता लगाया:

| अवधारणा | आपने क्या सीखा |
|---------|-----------------|
| **क्लाइंट** | `AzureAIProjectAgentProvider` क्रेडेंशियल-आधारित प्रमाणीकरण के साथ Azure OpenAI से कनेक्ट होता है |
| **एजेंट** | `provider.create_agent()` एक मॉडल कनेक्शन को निर्देशों और नाम के साथ बंडल करता है |
| **उपकरण** | `@tool` डेकोरेटर एजेंट के लिए पायथन फ़ंक्शंस को कॉल करने के लिए एक्सपोज़ करता है |
| **सत्र** | `agent.create_session()` कई मोड़ों में बातचीत का इतिहास बनाए रखता है |

ये निर्माण ब्लॉक्स मिलकर ऐसे एजेंट बनाते हैं जो प्राकृतिक बातचीत कर सकते हैं, बाहरी फ़ंक्शंस को कॉल कर सकते हैं, और संदर्भ बनाए रख सकते हैं — जो बाद के पाठों में अधिक उन्नत एजेंटिक पैटर्न के लिए आधार है।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यह दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके अनूदित किया गया है। जबकि हम सटीकता के लिए प्रयासरत हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या गलतियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए पेशेवर मानव अनुवाद की सलाह दी जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम जिम्मेदार नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
